# Data Cleaning and Visualization Project

This notebook follows the Google Colab workflow: load the employee CSV, inspect and clean the data, remove salary outliers, encode departments, validate and save the result, then create four visualizations.

## Step 1: Load the Dataset

In [ ]:
import os
import pandas as pd

FILE_NAME = 'sample_data_cleaning_project - Sample_data_cleaning_project.csv'
OUTPUT_FILE = 'cleaned_data.csv'
if not os.path.isfile(FILE_NAME):
    raise FileNotFoundError(f'Dataset not found: {FILE_NAME}. Place the CSV file in the same folder as this notebook.')
data = pd.read_csv(FILE_NAME)
if data.empty:
    raise ValueError('The dataset is empty.')
print('Dataset loaded successfully.')
print(f'Rows: {len(data)}')
print(f'Columns: {len(data.columns)}')
print('\nFirst 5 rows:')
print(data.head())

## Step 2: Identify Missing Values

In [ ]:
print('Missing values count:')
print(data.isnull().sum())
print('\nTotal missing values:', int(data.isnull().sum().sum()))

## Step 3: Clean Text and Handle Missing Values

In [ ]:
text_columns = data.select_dtypes(include='object').columns
for column in text_columns:
    data[column] = data[column].astype('string').str.strip()
data['Name'] = data['Name'].str.title()
data['Department'] = data['Department'].str.title()
data['Age'] = pd.to_numeric(data['Age'], errors='coerce')
data['Salary'] = pd.to_numeric(data['Salary'], errors='coerce')
for column in ['Age', 'Salary']:
    if data[column].isna().any():
        median_value = data[column].median()
        if pd.isna(median_value):
            raise ValueError(f'{column} has no valid numeric values.')
        data[column] = data[column].fillna(median_value)
data['Join_Date'] = pd.to_datetime(data['Join_Date'], errors='coerce')
invalid_dates = int(data['Join_Date'].isna().sum())
data = data.dropna(subset=['Join_Date']).copy()
print('Missing values after treatment:')
print(data.isnull().sum())
print(f'Invalid/missing dates removed: {invalid_dates}')

## Step 4: Check and Remove Duplicate Rows

In [ ]:
duplicates_before = int(data.duplicated().sum())
data = data.drop_duplicates().copy()
print('Exact duplicate rows removed:', duplicates_before)
print('Duplicate rows remaining:', int(data.duplicated().sum()))

## Step 5: Handle Salary Outliers

In [ ]:
Q1 = data['Salary'].quantile(0.25)
Q3 = data['Salary'].quantile(0.75)
IQR = Q3 - Q1
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR
outlier_mask = (data['Salary'] < lower_bound) | (data['Salary'] > upper_bound)
outlier_count = int(outlier_mask.sum())
data = data.loc[~outlier_mask].copy()
print(f'Q1: {Q1:.2f}')
print(f'Q3: {Q3:.2f}')
print(f'IQR: {IQR:.2f}')
print(f'Lower bound: {lower_bound:.2f}')
print(f'Upper bound: {upper_bound:.2f}')
print('Salary outliers removed:', outlier_count)

## Step 6: Convert Data Types

In [ ]:
data['Age'] = data['Age'].round().astype(int)
data['Salary'] = data['Salary'].astype(float)
data['Join_Date'] = pd.to_datetime(data['Join_Date'])
print('Final data types:')
print(data.dtypes)

## Step 7: Encode Categorical Variables

In [ ]:
data = pd.get_dummies(data, columns=['Department'], prefix='Department', drop_first=False, dtype=int)
department_columns = [column for column in data.columns if column.startswith('Department_')]
print('Encoded department columns:')
print(department_columns)
print('\nCleaned data:')
print(data.head())

## Step 8: Validate the Cleaned Dataset

In [ ]:
print('Final dataset shape:', data.shape)
print('\nMissing values:')
print(data.isnull().sum())
print('\nDuplicate rows:', int(data.duplicated().sum()))
if data.empty or data.isnull().sum().sum() != 0 or data.duplicated().any():
    raise ValueError('Validation failed: the cleaned dataset is not valid.')
print('\nValidation successful: dataset is clean.')

## Step 9: Save the Cleaned Dataset

In [ ]:
data.to_csv(OUTPUT_FILE, index=False)
if not os.path.isfile(OUTPUT_FILE):
    raise IOError('The cleaned dataset was not saved successfully.')
print(f'Cleaned dataset saved successfully as: {OUTPUT_FILE}')

## Step 10: Load the Cleaned Dataset for Visualization

In [ ]:
import matplotlib.pyplot as plt
cleaned_data = pd.read_csv(OUTPUT_FILE)
print('Cleaned dataset loaded for visualization.')
print(cleaned_data.head())

## Visualization 1: Department Distribution

In [ ]:
department_columns = [column for column in cleaned_data.columns if column.startswith('Department_')]
department_counts = cleaned_data[department_columns].sum().sort_values(ascending=False)
department_counts.index = department_counts.index.str.replace('Department_', '', regex=False)
plt.figure(figsize=(8, 6))
department_counts.plot(kind='bar')
plt.title('Department Distribution')
plt.xlabel('Department')
plt.ylabel('Number of Employees')
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig('visualizations/department_distribution.svg', bbox_inches='tight')
plt.savefig('visualizations/department_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

## Visualization 2: Salary Distribution

In [ ]:
plt.figure(figsize=(8, 6))
plt.hist(cleaned_data['Salary'], bins=10, edgecolor='black')
plt.title('Salary Distribution')
plt.xlabel('Salary')
plt.ylabel('Frequency')
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('visualizations/salary_distribution.svg', bbox_inches='tight')
plt.savefig('visualizations/salary_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

## Visualization 3: Salary vs. Age

In [ ]:
plt.figure(figsize=(8, 6))
plt.plot(cleaned_data['Age'], cleaned_data['Salary'], marker='o', linestyle='-')
plt.title('Salary vs. Age')
plt.xlabel('Age')
plt.ylabel('Salary')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('visualizations/salary_vs_age.svg', bbox_inches='tight')
plt.savefig('visualizations/salary_vs_age.png', dpi=150, bbox_inches='tight')
plt.show()

## Visualization 4: Salary vs. Age by Department

In [ ]:
plt.figure(figsize=(8, 6))
for column in department_columns:
    mask = cleaned_data[column].eq(1)
    department_name = column.replace('Department_', '')
    plt.scatter(cleaned_data.loc[mask, 'Salary'], cleaned_data.loc[mask, 'Age'], alpha=0.7, label=department_name)
plt.title('Salary vs. Age by Department')
plt.xlabel('Salary')
plt.ylabel('Age')
plt.grid(True, alpha=0.3)
plt.legend(title='Department')
plt.tight_layout()
plt.savefig('visualizations/salary_vs_age_by_department.svg', bbox_inches='tight')
plt.savefig('visualizations/salary_vs_age_by_department.png', dpi=150, bbox_inches='tight')
plt.show()